```{contents}
```
## Gradient Accumulation

### Motivation and Intuition

Modern deep learning models often require **large batch sizes** for stable optimization and better gradient estimates. However, GPU memory is limited.
**Gradient accumulation** allows you to simulate a large batch size by splitting it into multiple smaller mini-batches, accumulating their gradients, and performing a weight update only after several forward–backward passes.

**Key idea:**
Instead of:

```
Forward → Backward → Update (once per batch)
```

Do:

```
Forward → Backward (N times, accumulate gradients) → Update once
```

This achieves the effect of training with a batch size of:

```
effective_batch_size = mini_batch_size × accumulation_steps
```

---

### Mathematical Perspective

Standard SGD update:

$$
\theta \leftarrow \theta - \eta \cdot \nabla_{\theta} \mathcal{L}(x_1, \dots, x_B)
$$

With accumulation over $k$ steps:

$$
\nabla_{\theta} \approx \frac{1}{k} \sum_{i=1}^{k} \nabla_{\theta} \mathcal{L}_i
$$

This approximates the gradient of a larger batch while staying within memory limits.

---

### Training Workflow with Gradient Accumulation

| Step     | Operation                               |
| -------- | --------------------------------------- |
| Forward  | Compute loss on small mini-batch        |
| Backward | Accumulate gradients in `.grad` buffers |
| Repeat   | Do this `accumulation_steps` times      |
| Update   | Optimizer step once                     |
| Reset    | Zero gradients                          |

---

### Practical Use Cases

| Problem                                   | Solution                           |
| ----------------------------------------- | ---------------------------------- |
| GPU memory insufficient for large batches | Gradient accumulation              |
| Noisy gradients from small batches        | Accumulated gradient stabilization |
| Large transformer training                | Standard practice                  |

---

### PyTorch Demonstration

```python
model = MyModel().cuda()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = torch.nn.CrossEntropyLoss()

accum_steps = 4
optimizer.zero_grad()

for step, (x, y) in enumerate(loader):
    x, y = x.cuda(), y.cuda()

    outputs = model(x)
    loss = criterion(outputs, y)
    
    # Normalize loss to keep gradient scale consistent
    loss = loss / accum_steps
    loss.backward()

    if (step + 1) % accum_steps == 0:
        optimizer.step()
        optimizer.zero_grad()
```

**Effective batch size = batch_size × accum_steps**

---

### Why Loss Scaling is Required

Without dividing the loss:

* Gradients grow by `accum_steps`
* Learning rate becomes unintentionally larger
* Training becomes unstable

Loss normalization preserves correct gradient magnitude.

---

### Comparison with True Large Batch Training

| Property              | Large Batch       | Accumulated     |
| --------------------- | ----------------- | --------------- |
| Memory usage          | High              | Low             |
| Optimization behavior | Identical         | Identical       |
| Compute time          | Faster per update | Slightly slower |
| Stability             | High              | High            |

---

### Variants and Extensions

**Dynamic accumulation:**
Change accumulation steps during training based on available memory.

**Distributed accumulation:**
Combine with Distributed Data Parallel (DDP) for multi-GPU training.

**Mixed precision + accumulation:**
Common in large-scale training for additional memory savings.

---

### Failure Modes and Remediation

| Issue                     | Cause                              | Fix                        |
| ------------------------- | ---------------------------------- | -------------------------- |
| Exploding gradients       | Forgot loss normalization          | Divide loss by accum_steps |
| Slow convergence          | Too many accumulation steps        | Reduce accumulation        |
| Incorrect learning rate   | Mismatch with effective batch size | Scale LR accordingly       |
| Optimizer update mismatch | Skipping zero_grad                 | Ensure correct reset       |

---

### When Not to Use Gradient Accumulation

* When GPU memory is sufficient for full batch
* When training speed is more critical than memory
* For very small models where overhead dominates

---

### Summary

Gradient accumulation is a fundamental technique for scaling deep learning training under memory constraints. It preserves the optimization behavior of large batches while enabling training on limited hardware, making it essential for modern large-model workflows.
